# Vintage Delinquency & Net Loss Rate Analysis
### Consumer Credit / BNPL Portfolio — Cohort Performance Dashboard
**Author:** Denzel C. Seals | Senior Data Analyst  
**Date:** 2024  
**Stack:** Python · Pandas · Plotly · Seaborn

---

## Executive Summary

This analysis examines the delinquency and loss performance of a synthetic consumer credit portfolio across **8 origination vintages (2022-Q1 through 2023-Q4)**. Each vintage represents accounts originated within a calendar quarter, tracked monthly for up to 24 months on book (MOB).

**Key findings:**

- **Later 2023 vintages show elevated 90+ DPD and net loss rates** relative to 2022 cohorts at the same loan age, consistent with a tightening macro environment
- **30 DPD rates peak between MOB 6–10** across most vintages before resolving through cure or roll-forward
- **Portfolio average cumulative net loss rate reaches ~8–11% by MOB 18**, with meaningful spread across risk tiers
- **Average months to charge-off** is approximately 9–11 months — consistent with typical BNPL/consumer credit charge-off timing

---

## Analytical Framework

| Component | Description |
|---|---|
| **Vintage definition** | Origination quarter (e.g. 2023-Q1 = Jan–Mar 2023 originations) |
| **Observation unit** | Account × calendar month |
| **DPD buckets** | Current, 30, 60, 90, 120+, Charged Off |
| **Charge-off trigger** | Account reaches 120+ DPD → charged off next month |
| **Net loss rate** | Cumulative net losses ÷ original cohort balance at origination |
| **Recovery assumption** | 15% of charged-off balance, applied at MOB + 6 |

---


## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from data_generator import main as generate_data
from feature_engineering import (
    label_dpd_buckets,
    compute_roll_rates,
    compute_loss_build,
    compare_vintages_at_mob,
    portfolio_summary_stats,
)
from visualization import (
    plot_vintage_dpd_curves,
    plot_dpd_dashboard,
    plot_roll_rate_heatmap,
    plot_net_loss_curves,
    plot_cohort_snapshot,
)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)


In [ ]:
# Generate synthetic data (skip if already generated)
import os
if not os.path.exists("../data/raw/portfolio_account_months.parquet"):
    print("Generating synthetic portfolio data...")
    generate_data()
else:
    print("Data already exists — loading from disk.")

df = pd.read_parquet("../data/raw/portfolio_account_months.parquet")
vintage_summary = pd.read_parquet("../data/processed/vintage_summary.parquet")

print(f"Account-month rows: {len(df):,}")
print(f"Vintage summary rows: {len(vintage_summary):,}")
print(f"Vintages: {sorted(df['vintage'].unique())}")


In [ ]:
# Preview raw data structure
df.head(10)


## 2. Portfolio Overview

Before diving into vintage curves, we establish the portfolio-level picture:
- Total accounts and origination balance
- Share of accounts that ever became delinquent
- Charge-off rates and average time-to-charge-off


In [ ]:
stats = portfolio_summary_stats(df, vintage_summary)

print("=" * 50)
print("PORTFOLIO SUMMARY")
print("=" * 50)
print(f"Total accounts:           {stats['total_accounts']:,}")
print(f"Total orig. balance:      ${stats['total_orig_balance_mm']:.1f}M")
print(f"Ever 30+ DPD:             {stats['pct_ever_delinquent']:.1f}%")
print(f"Charged off:              {stats['pct_charged_off']:.1f}%")
print(f"Avg months to charge-off: {stats['avg_months_to_charge_off']:.1f}")
print()
print("Max Net Loss Rate by Vintage:")
for vintage, rate in stats['net_loss_rate_by_vintage'].items():
    print(f"  {vintage}: {rate*100:.2f}%")


In [ ]:
# Cohort sizes and origination balance by vintage
cohort_profile = (
    df[df["months_on_book"] == 1]
    .groupby(["vintage", "risk_tier"])
    .agg(
        accounts=("account_id", "nunique"),
        orig_balance_mm=("origination_balance", lambda x: x.sum() / 1e6),
        avg_balance=("origination_balance", "mean"),
    )
    .round(2)
    .reset_index()
)
cohort_profile


## 3. Vintage Delinquency Curves

The vintage DPD curve is the foundational view in credit portfolio monitoring.
Each line represents a single origination cohort tracked from month 1 through
the observation horizon. The x-axis is **months on book (MOB)** — not calendar date —
which removes the portfolio mix effect and isolates true cohort aging behavior.

**What to look for:**
- **Early peak**: Does a newer vintage peak earlier or higher than prior cohorts?
- **Cure vs. roll**: High 30 DPD with low 60 DPD suggests strong cure rates
- **Plateau level**: Where does the 90 DPD rate stabilize? This informs charge-off projections


In [ ]:
# 30 DPD curves — all vintages
fig = plot_vintage_dpd_curves(vintage_summary, dpd_metric="dpd_30_rate", title_suffix="30 DPD")
fig.show()


In [ ]:
# 60 DPD curves
fig = plot_vintage_dpd_curves(vintage_summary, dpd_metric="dpd_60_rate", title_suffix="60 DPD")
fig.show()


In [ ]:
# 90+ DPD curves — closest leading indicator of loss
fig = plot_vintage_dpd_curves(vintage_summary, dpd_metric="dpd_90_rate", title_suffix="90+ DPD")
fig.show()


In [ ]:
# Full dashboard — all DPD buckets + net loss rate in one view
fig = plot_dpd_dashboard(vintage_summary)
fig.show()


**Observations:**
- 2023-Q3 and 2023-Q4 vintages show noticeably higher 90+ DPD rates at MOB 6–12 relative to 2022 cohorts — consistent with a macro stress overlay applied in the data generator (simulating real 2023 BNPL credit tightening)
- 30 DPD rates peak around MOB 8–10, then decay as accounts either cure or roll to deeper delinquency
- The portfolio average (dashed) provides a natural benchmark for identifying above/below-average vintages


## 4. Roll Rate Analysis

A **roll rate matrix** shows the probability that an account in a given DPD bucket
transitions to each other bucket in the next month. This is central to:
- **Loss forecasting**: project future charge-offs from current delinquency stock
- **Cure rate monitoring**: track whether accounts are returning to current vs. rolling forward
- **Vintage-level stress testing**: compare roll matrices across cohorts

We compute the matrix across all vintages combined, then isolate a specific cohort for comparison.


In [ ]:
# Portfolio-wide roll rate matrix
roll_matrix_all = compute_roll_rates(df)
roll_matrix_all


In [ ]:
# Heatmap — portfolio-wide
ax = plot_roll_rate_heatmap(roll_matrix_all, vintage="All Vintages")


In [ ]:
# Compare: 2022-Q1 (seasoned, lower stress) vs 2023-Q4 (recent, higher stress)
roll_2022q1 = compute_roll_rates(df, vintage="2022-Q1")
roll_2023q4 = compute_roll_rates(df, vintage="2023-Q4")

print("2022-Q1 Roll Rates:")
display(roll_2022q1)

print("\n2023-Q4 Roll Rates:")
display(roll_2023q4)


In [ ]:
ax = plot_roll_rate_heatmap(roll_2022q1, vintage="2022-Q1")
ax = plot_roll_rate_heatmap(roll_2023q4, vintage="2023-Q4")


**Key interpretation:**
- Compare the `dpd_30 → dpd_60` transition rate: a rising roll rate here is an early warning signal
- Compare the `current → dpd_30` entry rate: this is the single most important leading indicator
- The 2023-Q4 matrix should show higher roll-forward rates, reflecting the macro stress assumption


## 5. Net Loss Rate by Vintage

The cumulative net loss rate measures total credit losses (after recoveries) as a
percentage of the original origination balance. This is the ultimate P&L metric
for a credit portfolio — it determines whether a vintage is profitable given its yield.

**Formula:**  
`Net Loss Rate = (Cumulative Gross Losses − Cumulative Recoveries) ÷ Original Cohort Balance`

**Recovery assumption:** 15% of charged-off balance, applied 6 months after charge-off.


In [ ]:
fig = plot_net_loss_curves(vintage_summary)
fig.show()


In [ ]:
# Loss build schedule — how fast losses emerge per vintage
loss_build = compute_loss_build(vintage_summary)

# Show loss rate at MOB 6, 12, 18, 24 for each vintage
loss_snapshots = loss_build[loss_build["months_on_book"].isin([6, 12, 18, 24])].pivot_table(
    index="vintage",
    columns="months_on_book",
    values="net_loss_rate",
).round(4) * 100

loss_snapshots.columns = [f"MOB {c}" for c in loss_snapshots.columns]
loss_snapshots.style.format("{:.2f}%").background_gradient(cmap="YlOrRd", axis=None)


## 6. Period-over-Period Comparison

To benchmark vintages fairly, we compare them **at the same loan age** (fixed MOB).
This removes the calendar-date bias that makes newer vintages appear better simply
because they haven't had time to season.

Below we snapshot all vintages at **MOB 12** and rank by net loss rate.
Vintages above the portfolio average are flagged in red.


In [ ]:
# Snapshot at MOB 12 — net loss rate
comparison_12 = compare_vintages_at_mob(vintage_summary, mob=12, metric="net_loss_rate")
comparison_12


In [ ]:
fig = plot_cohort_snapshot(comparison_12, mob=12, metric="net_loss_rate")
fig.show()


In [ ]:
# Snapshot at MOB 12 — 90+ DPD rate
comparison_90 = compare_vintages_at_mob(vintage_summary, mob=12, metric="dpd_90_rate")
fig = plot_cohort_snapshot(comparison_90, mob=12, metric="dpd_90_rate")
fig.show()


## 7. Risk Tier Segmentation

Vintage-level aggregates can mask meaningful differences in underwriting quality.
Segmenting by risk tier (Prime / Near-Prime / Subprime) reveals:
- Whether a deteriorating vintage is driven by mix shift or universal performance decline
- Which tiers are contributing disproportionate losses


In [ ]:
# DPD rate by vintage and risk tier at MOB 12
tier_snapshot = (
    df[df["months_on_book"] == 12]
    .groupby(["vintage", "risk_tier"])
    .agg(
        accounts=("account_id", "nunique"),
        dpd_30_ct=("is_30_dpd", "sum"),
        dpd_90_ct=("is_90_dpd", "sum"),
        co_ct=("is_charged_off", "sum"),
    )
    .reset_index()
)

# Cohort sizes for rate calculation
cohort_size = df[df["months_on_book"] == 1].groupby(["vintage", "risk_tier"])["account_id"].nunique().rename("cohort_size").reset_index()
tier_snapshot = tier_snapshot.merge(cohort_size, on=["vintage", "risk_tier"])
tier_snapshot["dpd_30_rate"] = (tier_snapshot["dpd_30_ct"] / tier_snapshot["cohort_size"] * 100).round(2)
tier_snapshot["dpd_90_rate"] = (tier_snapshot["dpd_90_ct"] / tier_snapshot["cohort_size"] * 100).round(2)
tier_snapshot["co_rate"]     = (tier_snapshot["co_ct"]     / tier_snapshot["cohort_size"] * 100).round(2)

tier_snapshot[["vintage", "risk_tier", "cohort_size", "dpd_30_rate", "dpd_90_rate", "co_rate"]]


In [ ]:
# Pivot: 90+ DPD rate at MOB 12 by vintage × risk tier
pivot = tier_snapshot.pivot_table(index="vintage", columns="risk_tier", values="dpd_90_rate")
pivot.style.format("{:.2f}%").background_gradient(cmap="YlOrRd", axis=None)


## 8. Conclusions & Analyst Commentary

### Summary of Findings

1. **2022 vintages are outperforming 2023 cohorts** at equivalent loan ages across all DPD buckets. The macro stress factor applied to 2023 originations (simulating tighter credit environment) is visible by MOB 6 and widens through MOB 12.

2. **Roll rates from 30 → 60 DPD** are the most actionable early warning signal. The 2023-Q4 roll matrix shows materially higher 30 → 60 transition rates compared to 2022-Q1, suggesting fewer cures and greater loss severity for newer cohorts.

3. **Net loss rates are on track to mature around 8–12%** across vintages, with later 2023 cohorts potentially reaching the upper end of that range if macro conditions do not improve.

4. **Subprime tier accounts** drive a disproportionate share of 90+ DPD and charge-offs relative to their share of origination volume — standard in BNPL portfolios, but worth monitoring for mix shift in newer vintages.

### What Would Change This View

- A cure-rate improvement program targeted at 30–60 DPD accounts could reduce projected net loss rates by 1–2 percentage points
- Tighter underwriting in 2024 (lower subprime mix, income verification) would be expected to show up as lower 30 DPD entry rates by MOB 3–6
- Recovery rates above 15% (e.g., through more aggressive collections investment) would improve net loss rates on charged-off cohorts

### Extensions & Next Steps

| Extension | Analytical Value |
|---|---|
| Logistic regression loss forecasting | Project MOB 18–24 net loss rates from current MOB 6–12 delinquency |
| Yield vs. loss trade-off | Layer in interest income assumptions to compute cohort-level margin |
| Stress testing | Shock cure rates by ±20% to bound loss projections under pessimistic and optimistic scenarios |
| Live portfolio monitoring | Replace synthetic data with real portfolio data and parameterize for monthly refresh |

---
*This analysis was built as a portfolio demonstration of vintage credit analytics methodology. All data is synthetic and generated programmatically — see `src/data_generator.py` for full parameter controls.*
